# Partie 1 - Test des descripteurs sans IA 
1. HOG, descripteur de forme globale
2. SIFT et ORB, descripteur de points locaux

**Justification :**

On évite les descripteurs de couleurs car ca n'a pas de réel sens de comparer des voitures sur leur couleur.

Même raisonnement pour les descripteurs de textures, cela n'a pas de sens pour comparer différentes voitures.

Pour ce qui est de HOG, SIFT et ORB, ce sont des descripteurs qui pourront surement capter les spécificités des marques/modèles. 

## 1.1 Implémentation de HOG (Histogram of Oriented Gradients)
Nous allons extraire les caractéristiques de forme. Les images sont redimensionnées en 128x128 et converties en niveaux de gris pour uniformiser la taille du descripteur.

In [ ]:
import os
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = "../data/raw/Cars"
all_images = [f for f in os.listdir(DATA_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

def extract_hog_features(img_path, target_size=(128, 128), visualize=False):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    
    img_resized = cv2.resize(img, target_size)
    
    if visualize:
        features, hog_image = hog(img_resized, orientations=9, pixels_per_cell=(8, 8),
                                  cells_per_block=(2, 2), visualize=True)
        return features, hog_image, img_resized
    else:
        features = hog(img_resized, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), visualize=False)
        return features

fig, axes = plt.subplots(3, 2, figsize=(8, 10))
for i in range(3):
    img_path = os.path.join(DATA_DIR, all_images[i])
    features, hog_img, img_res = extract_hog_features(img_path, visualize=True)
    
    axes[i, 0].imshow(img_res, cmap='gray')
    axes[i, 0].set_title(f"Originale (Grayscale)\n{all_images[i]}")
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(hog_img, cmap='gray')
    axes[i, 1].set_title(f"HOG Features\nDimensions: {features.shape[0]}")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

On voit que, sur les images avec un fond neutre, on perçoit bien les caractéristiques de forme de la voiture. Par ailleurs, dès que le fond se complexifie, il y a plusieurs formes différentes qui sont mises en avant, et percevoir la voiture devient de plus en plus compliqué.

In [ ]:
print(f"Début de l'indexation HOG pour {len(all_images)} images...")

db_features = []
db_paths = []
db_classes = []

start_index_time = time.time()

for img_name in all_images:
    img_path = os.path.join(DATA_DIR, img_name)
    feat = extract_hog_features(img_path)
    
    if feat is not None:
        db_features.append(feat)
        db_paths.append(img_path)
        
        parts = img_name.split('_')
        db_classes.append(f"{parts[0]}_{parts[1]}")

end_index_time = time.time()

db_features = np.array(db_features)
db_paths = np.array(db_paths)
db_classes = np.array(db_classes)

indexing_time = end_index_time - start_index_time
descriptor_size_mb = db_features.nbytes / (1024 * 1024)

print("-" * 40)
print(f"Temps total d'indexation : {indexing_time:.2f} secondes")
print(f"Taille de la base de descripteurs : {descriptor_size_mb:.2f} MB")
print(f"Dimension d'un descripteur HOG : {db_features.shape[1]}")
print("-" * 40)

In [ ]:
def evaluate_query(retrieved_labels, query_label, total_relevant, k):
    retrieved_k = retrieved_labels[:k]
    
    relevance = [1 if label == query_label else 0 for label in retrieved_k]
    
    relevant_retrieved = sum(relevance)
    
    precision = relevant_retrieved / k
    
    recall = relevant_retrieved / total_relevant if total_relevant > 0 else 0
    
    ap = 0.0
    relevant_count = 0
    for i, rel in enumerate(relevance):
        if rel == 1:
            relevant_count += 1
            ap += relevant_count / (i + 1)
            
    if relevant_count > 0:
        ap = ap / min(total_relevant, k) 
    else:
        ap = 0.0
        
    return precision, recall, ap

In [ ]:
QUERIES_GROUP_5 = [
    "0_1_BMW_X3_207.jpg",
    "0_0_BMW_Serie3Berline_74.jpg",
    "0_2_BMW_i8_299.jpg",
    "2_0_Volkswagen_Touareg_2822.jpg",
    "2_4_Volkswagen_Polo_3463.jpg",
    "2_9_Volkswagen_T-Roc_4209.jpg",
    "4_2_Opel_vivarofourgon_5999.jpg",
    "4_4_Opel_Insignatourer_6353.jpg",
    "4_9_Opel_zafiralife_6887.jpg",
    "6_0_Hyundai_Nexo_8282.jpg",
    "6_3_Hyundai_i10_8837.jpg",
    "6_5_Hyundai_i30_9125.jpg",
    "8_1_Ford_Puma_11276.jpg",
    "8_5_Ford_Explorer_11897.jpg",
    "8_6_Ford_Focus_11951.jpg"
]

metrics_50 = {'P': [], 'R': [], 'AP': []}
metrics_100 = {'P': [], 'R': [], 'AP': []}
metrics_max = {'P': [], 'R': [], 'AP': []}
search_times = []

for i, q_img in enumerate(QUERIES_GROUP_5):
    q_path = os.path.join(DATA_DIR, q_img)
    parts = q_img.split('_')
    q_class = f"{parts[0]}_{parts[1]}"
    
    total_relevant = sum(1 for c in db_classes if c == q_class)
    print(f"Max = {total_relevant}")

    start_search = time.time()
    q_feat = extract_hog_features(q_path)
    if q_feat is None: continue
        
    similarities = cosine_similarity(q_feat.reshape(1, -1), db_features)[0]
    
    top_indices = np.argsort(similarities)[::-1]
    end_search = time.time()
    
    search_times.append(end_search - start_search)
    
    retrieved_labels = [db_classes[idx] for idx in top_indices if db_paths[idx] != q_path]
    
    p_50, r_50, ap_50 = evaluate_query(retrieved_labels, q_class, total_relevant, k=50)
    metrics_50['P'].append(p_50)
    metrics_50['R'].append(r_50)
    metrics_50['AP'].append(ap_50)
    
    p_100, r_100, ap_100 = evaluate_query(retrieved_labels, q_class, total_relevant, k=100)
    metrics_100['P'].append(p_100)
    metrics_100['R'].append(r_100)
    metrics_100['AP'].append(ap_100)

    p_max, r_max, ap_max = evaluate_query(retrieved_labels, q_class, total_relevant, k=total_relevant)
    metrics_max['P'].append(p_max)
    metrics_max['R'].append(r_max)
    metrics_max['AP'].append(ap_max)
    
    print(f"Requête {i+1} | "
          f"P@50: {p_50*100:.1f}% | R@50: {r_50*100:.1f}% | AP@50: {ap_50*100:.1f}% | P@100: {p_100*100:.1f}% | R@100: {r_100*100:.1f}% | AP@100: {ap_100*100:.1f}% | P@Max: {p_max*100:.1f}% | R@Max: {r_max*100:.1f}% | AP@Max: {ap_max*100:.1f}%")

print("\n" + "=" * 40)
print("BILAN FINAL - DESCRIPTEUR HOG")
print("=" * 40)
print(f"Temps de recherche moyen  : {np.mean(search_times)*1000:.2f} ms / image")
print("\n--- Performances @50 ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_50['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_50['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_50['AP'])*100:.2f} %")

print("\n--- Performances @100 ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_100['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_100['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_100['AP'])*100:.2f} %")

print("\n--- Performances @Max ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_max['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_max['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_max['AP'])*100:.2f} %")

On voit en général que la précision moyenne est assez faible, signifiant que le descripteur HOG renvoie beaucoup d'erreurs. Par ailleurs, le recall moyen est aussi très faible, signifiant que les nombre de vraies images intéressantes retrouvées est très faible. C'est un descripteur qui peut ne pas bien marcher par après.

## 1.2 Implémentation de SIFT (Scale-Invariant Feature Transform)
SIFT est un descripteur local. Contrairement à HOG, il va détecter plusieurs points d'intérêt dans l'image et extraire un descripteur pour chacun d'eux. Pour comparer deux images, nous allons devoir "matcher" (associer) leurs points d'intérêt respectifs. Plus il y a de bons matchs, plus les images sont considérées comme similaires.

Note : Conformément aux consignes du projet, nous réduisons la résolution des images avant l'extraction SIFT pour limiter la complexité de calcul.## 1.2 Implémentation de SIFT (Scale-Invariant Feature Transform)

SIFT est un descripteur local. Contrairement à HOG, il va détecter plusieurs points d'intérêt dans l'image et extraire un descripteur pour chacun d'eux. 
Pour comparer deux images, nous allons devoir "matcher" (associer) leurs points d'intérêt respectifs. Plus il y a de bons matchs, plus les images sont considérées comme similaires.

*Note : Conformément aux consignes du projet, nous réduisons la résolution des images avant l'extraction SIFT pour limiter la complexité de calcul.*

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

sift = cv2.SIFT_create()

example_query = QUERIES_GROUP_5[14]
example_path = os.path.join(DATA_DIR, example_query)
example_img = cv2.imread(example_path, cv2.IMREAD_GRAYSCALE)

if example_img is not None:
    keypoints, _ = sift.detectAndCompute(example_img, None)
    img_with_keypoints = cv2.drawKeypoints(example_img, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    plt.figure(figsize=(6, 6))
    plt.imshow(img_with_keypoints, cmap='gray')
    plt.title(f"Points d'intérêt SIFT détectés\n{example_query}")
    plt.axis('off')
    plt.show()

def extract_sift_descriptors(image_path, max_size=256):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
        
    h, w = img.shape
    if max(h, w) > max_size:
        scale = max_size / max(h, w)
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
        
    _, descriptors = sift.detectAndCompute(img, None)
    return descriptors

print("Extraction des descripteurs SIFT pour la base de données...")
db_sift_features = {}

for db_file in tqdm(all_images):
    db_path = os.path.join(DATA_DIR, db_file)
    des = extract_sift_descriptors(db_path)
    if des is not None:
        db_sift_features[db_file] = des

In [ ]:
def extract_class_and_model(filename):
    basename = os.path.basename(filename)
    
    parts = basename.split('_')
    
    img_class = f"{parts[0]}_{parts[1]}"
    
    return img_class, parts

class_counts = {}

for db_file in all_images:
    db_class, _ = extract_class_and_model(db_file)
    class_counts[db_class] = class_counts.get(db_class, 0) + 1

print("Nombre d'images par classe dans la base :")
for cls, count in class_counts.items():
    print(f"Classe {cls} : {count} images")

In [ ]:
import time

metrics_50 = {'P': [], 'R': [], 'AP': []}
metrics_100 = {'P': [], 'R': [], 'AP': []}
metrics_max = {'P': [], 'R': [], 'AP': []}
search_times = []

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv2.FlannBasedMatcher(index_params, search_params)

print("=== DEBUT DE L'EVALUATION SIFT (GROUPE 05) ===")

for i, query in enumerate(QUERIES_GROUP_5):
    start_time = time.time()
    
    query_path = os.path.join(DATA_DIR, query)
    query_class, _ = extract_class_and_model(query)
    total_relevant = class_counts[query_class]
    
    query_des = extract_sift_descriptors(query_path)
    scores = [] 
    
    if query_des is not None and len(query_des) > 2:
        for db_file, db_des in db_sift_features.items():
            if db_des is None or len(db_des) < 2:
                continue
                
            try:
                matches = flann.knnMatch(query_des, db_des, k=2)
                
                good_matches = 0
                for m_n in matches:
                    if len(m_n) == 2:
                        m, n = m_n
                        if m.distance < 0.7 * n.distance:
                            good_matches += 1
                            
                db_class, _ = extract_class_and_model(db_file)
                scores.append((good_matches, db_class))
            except Exception:
                pass

    scores.sort(key=lambda x: x[0], reverse=True)
    retrieved_labels = [label for _, label in scores]
    
    search_times.append(time.time() - start_time)
    
    p_50, r_50, ap_50 = evaluate_query(retrieved_labels, query_class, total_relevant, k=50)
    metrics_50['P'].append(p_50)
    metrics_50['R'].append(r_50)
    metrics_50['AP'].append(ap_50)
    
    p_100, r_100, ap_100 = evaluate_query(retrieved_labels, query_class, total_relevant, k=100)
    metrics_100['P'].append(p_100)
    metrics_100['R'].append(r_100)
    metrics_100['AP'].append(ap_100)
    
    p_max, r_max, ap_max = evaluate_query(retrieved_labels, query_class, total_relevant, k=total_relevant)
    metrics_max['P'].append(p_max)
    metrics_max['R'].append(r_max)
    metrics_max['AP'].append(ap_max)
    
    print(f"Requête {i+1} | "
          f"P@50: {p_50*100:.1f}% | R@50: {r_50*100:.1f}% | AP@50: {ap_50*100:.1f}% | P@100: {p_100*100:.1f}% | R@100: {r_100*100:.1f}% | AP@100: {ap_100*100:.1f}% | P@Max: {p_max*100:.1f}% | R@Max: {r_max*100:.1f}% | AP@Max: {ap_max*100:.1f}%")

print("\n" + "=" * 40)
print("BILAN FINAL - DESCRIPTEUR SIFT")
print("=" * 40)
print(f"Temps de recherche moyen  : {np.mean(search_times)*1000:.2f} ms / image")

print("\n--- Performances @50 ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_50['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_50['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_50['AP'])*100:.2f} %")

print("\n--- Performances @100 ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_100['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_100['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_100['AP'])*100:.2f} %")

print("\n--- Performances @Max ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_max['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_max['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_max['AP'])*100:.2f} %")

## 1.3 Implémentation de ORB (Oriented Fast and Rotated Brief)
ORB cherche des points d'intérêts très précis et crée une signature unique pour chacun d'eux. Ce descripteur va d'abord chercher des zones qui se démarquent de leur environnement en utilisant un détecteur FAST. Ce detecteur prend un pixel, regarde un cercle de 16 pixels autour de lui et si une majorité de ces pixels sont beaucoup plus clairs ou foncés, il a trouvé un point d'intérêt. ORB calcule également l'orientation de ce point. Le descripteur BRIEF prend des paires de pixels autour du points d'intérêt et compare leur luminosité, afin de les noter 1 ou 0. Chaque point d'intérêt se retoruve alors avec une signature binaire.



In [ ]:
import cv2 as cv
import numpy as np
import os
from matplotlib import pyplot as plt

#Création du detecteur ORB
orb = cv.ORB_create()

#Image d'exemple
example_query = QUERIES_GROUP_5[14]
example_path = os.path.join(DATA_DIR, example_query)
example_img = cv2.imread(example_path, cv2.IMREAD_GRAYSCALE)
 
if example_img is not None:
     
    
    keypoints = orb.detect(example_img,None)
     
    #Descripteurs avec ORB
    keypoints, des = orb.compute(example_img, keypoints)
     
    #Dessiner uniquement les positions des keypoints, pas la taille ni l'orientation
    img2 = cv.drawKeypoints(example_img, keypoints, None, color=(0,255,0), flags=0)
    plt.figure(figsize=(6,6))
    plt.imshow(img2)
    plt.title(f"Points d'intérêts ORB détectés \n{example_query}")
    plt.axis('off')
    plt.show()

In [ ]:
def extract_ORB_descriptors(image_path, max_size=256):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None :
        return None
    h, w = img.shape
    if max(h,w) > max_size:
        scale = max_size/max(h,w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)))
    keypoints = orb.detect(img,None)
    keypoints, descriptors = orb.compute(img, keypoints)
    return descriptors
print("Extraction des descripteurs ORB pour la base de données...")
db_orb_features = {}
for db_file in tqdm(all_images):
    db_path = os.path.join(DATA_DIR, db_file)
    des = extract_ORB_descriptors(db_path)
    if des is not None:
        db_orb_features[db_file] = des

In [ ]:
import time 
metrics_50 = {'P': [], 'R': [], 'AP': []}
metrics_100 = {'P': [], 'R': [], 'AP': []}
metrics_max = {'P': [], 'R': [], 'AP': []}
search_times = []

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
bf = cv2.BFMatcher(cv2.NORM_HAMMING)

print("=== DEBUT DE L'EVALUATION ORB (GROUPE 05) ===")
for i, query in enumerate(QUERIES_GROUP_5):
    start_time = time.time()
    
    query_path = os.path.join(DATA_DIR, query)
    query_class, _ = extract_class_and_model(query)
    total_relevant = class_counts[query_class]
    
    query_des = extract_ORB_descriptors(query_path)
    scores = []
    if query_des is not None and len(query_des) > 2:
        for db_file, db_des in db_orb_features.items():
            if db_des is None or len(db_des) < 2:
                continue
                
            try:
                matches = bf.knnMatch(query_des, db_des, k=2)
                
                good_matches = 0
                for m_n in matches:
                    if len(m_n) == 2:
                        m, n = m_n
                        if m.distance < 0.75 * n.distance:
                            good_matches += 1
                            
                db_class, _ = extract_class_and_model(db_file)
                scores.append((good_matches, db_class))
            except Exception as e:
                print(f"Erreur avec l'image {db_file}: {e}")
            
    scores.sort(key=lambda x: x[0], reverse=True)
    retrieved_labels = [label for _, label in scores]
    
    search_times.append(time.time() - start_time)
    
    p_50, r_50, ap_50 = evaluate_query(retrieved_labels, query_class, total_relevant, k=50)
    metrics_50['P'].append(p_50)
    metrics_50['R'].append(r_50)
    metrics_50['AP'].append(ap_50)
    
    p_100, r_100, ap_100 = evaluate_query(retrieved_labels, query_class, total_relevant, k=100)
    metrics_100['P'].append(p_100)
    metrics_100['R'].append(r_100)
    metrics_100['AP'].append(ap_100)
    
    p_max, r_max, ap_max = evaluate_query(retrieved_labels, query_class, total_relevant, k=total_relevant)
    metrics_max['P'].append(p_max)
    metrics_max['R'].append(r_max)
    metrics_max['AP'].append(ap_max)
    
    print(f"Requête {i+1} | "
          f"P@50: {p_50*100:.1f}% | R@50: {r_50*100:.1f}% | AP@50: {ap_50*100:.1f}% | P@100: {p_100*100:.1f}% | R@100: {r_100*100:.1f}% | AP@100: {ap_100*100:.1f}% | P@Max: {p_max*100:.1f}% | R@Max: {r_max*100:.1f}% | AP@Max: {ap_max*100:.1f}%")

print("\n" + "=" * 40)
print("BILAN FINAL - DESCRIPTEUR ORB")
print("=" * 40)
print(f"Temps de recherche moyen  : {np.mean(search_times)*1000:.2f} ms / image")

print("\n--- Performances @50 ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_50['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_50['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_50['AP'])*100:.2f} %")

print("\n--- Performances @100 ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_100['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_100['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_100['AP'])*100:.2f} %")

print("\n--- Performances @Max ---")
print(f"Precision moyenne (mP)    : {np.mean(metrics_max['P'])*100:.2f} %")
print(f"Recall moyen (mR)         : {np.mean(metrics_max['R'])*100:.2f} %")
print(f"Mean Avg Precision (mAP)  : {np.mean(metrics_max['AP'])*100:.2f} %")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['Top 50 (@50)', 'Top 100 (@100)', 'Tout (@Max)']

mP = [np.mean(metrics_50['P'])*100, np.mean(metrics_100['P'])*100, np.mean(metrics_max['P'])*100]
mR = [np.mean(metrics_50['R'])*100, np.mean(metrics_100['R'])*100, np.mean(metrics_max['R'])*100]

x = np.arange(len(labels))
largeur_barre = 0.35 

fig, ax = plt.subplots(figsize=(10, 6))

barres_precision = ax.bar(x - largeur_barre/2, mP, largeur_barre, label='Précision (mP)', color='#3498db')
barres_recall = ax.bar(x + largeur_barre/2, mR, largeur_barre, label='Rappel (mR)', color='#e74c3c')

ax.set_ylabel('Scores (%)', fontsize=12, fontweight='bold')
ax.set_title('Évolution de la Précision et du Rappel (Descripteur ORB)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 105) 
ax.legend(fontsize=11)


ax.grid(axis='y', linestyle='--', alpha=0.7)

for barre in barres_precision + barres_recall:
    hauteur = barre.get_height()
    ax.annotate(f'{hauteur:.1f}%',
                xy=(barre.get_x() + barre.get_width() / 2, hauteur),
                xytext=(0, 3), 
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()